In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df = spark.table("proj.aviation.flights_silver")

gold_airline_df = (
    silver_df.groupBy("FlightDate", "Reporting_Airline")
    .agg(
        count("*").alias("total_flights"),
        count(when(col("Cancelled") == 1, 1)).alias("cancelled_flights"),
        count(when(col("Diverted") == 1, 1)).alias("diverted_flights"),
        
        count(when(col("ArrDel15") == 0, 1)).alias("on_time_flights"),
        round(
            (count(when(col("ArrDel15") == 0, 1)) / count("*")) * 100, 2
        ).alias("on_time_pct"),
        
        round(avg("ArrDelay"), 2).alias("avg_arr_delay_mins"),
        round(sum("CarrierDelay"), 2).alias("total_carrier_delay_mins"),
        round(sum("WeatherDelay"), 2).alias("total_weather_delay_mins"),
        round(sum("NASDelay"), 2).alias("total_nas_delay_mins"),
        round(sum("SecurityDelay"), 2).alias("total_security_delay_mins"),
        round(sum("LateAircraftDelay"), 2).alias(
            "total_late_aircraft_delay_mins"
        ),
    )
    .withColumn("_gold_loaded_dttm", current_timestamp())
)



In [0]:
gold_airline_df.write.format("delta").mode("overwrite").saveAsTable(
    "proj.aviation.gold_airline_performance_daily"
)

In [0]:
gold_route_df = (
    silver_df.groupBy("Year", "Month", "Origin", "Dest")
    .agg(
        count("*").alias("total_flights"),
        count(when(col("Cancelled") == 1, 1)).alias("cancelled_flights"),
        round(avg("ArrDelay"), 2).alias("avg_arr_delay_mins"),
        round(avg("AirTime"), 2).alias("avg_air_time_mins"),
        round(avg("Distance"), 2).alias("avg_distance_miles"),
    )
    .withColumn("_gold_loaded_dttm", current_timestamp())
)



In [0]:

gold_route_df.write.format("delta").mode("overwrite").saveAsTable(
    "proj.aviation.gold_route_performance_monthly"
)